# Refusal vs Scratch Sampling Analysis - Multiple Tactics and Batches\n\nThis notebook analyzes whether sampling from scratch is equivalent to sampling after a refusal by comparing the distribution of scores for:\n1. **First responses** (sampling from scratch): The very first response in each conversation\n2. **Subsequent responses** (sampling after refusal): Responses that come after at least one \"refused\" score\n\n**Data**: Both batch6A and batch6B, both direct_request and command tactics, single-turn conversations only\n\n**Analysis**: For each JSONL file with score sequence like [\"refused\", \"refused\", 0.7], the first \"refused\" goes to the scratch distribution, while the second \"refused\" and 0.7 go to the after-refusal distribution.\n\n**Creates 5 plots**:\n- 4 individual plots for [direct_request, command] x [batch6A, batch6B]\n- **1 combined plot** pooling all data for maximum statistical power"

In [15]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path
from typing import Dict, List, Optional, Tuple
import seaborn as sns
from scipy import stats

# Set style for better plots
plt.style.use('default')
sns.set_palette("husl")

# Configuration
BATCH_PATHS = {
    'batch6A': 'clean_results/final_runs/batch6A',
    'batch6B': 'clean_results/final_runs/batch6B'
}
TACTICS = ['direct_request', 'command']
BINS = 20

In [16]:
def load_jsonl_metadata(file_path: str) -> Optional[Dict]:
    """
    Load a JSONL file and extract metadata from the first line.
    """
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            # Read the first line which contains metadata
            first_line = file.readline().strip()
            if first_line:
                metadata = json.loads(first_line)
                return {
                    'jailbreak_tactic': metadata.get('jailbreak_tactic'),
                    'test_case': metadata.get('test_case'),
                    'target_model': metadata.get('target_model'),
                    'turn_type': metadata.get('turn_type'),
                    'timestamp': metadata.get('timestamp'),
                    'sample_id': metadata.get('sample_id', 1)
                }
    except (json.JSONDecodeError, FileNotFoundError, Exception) as e:
        print(f"Error processing file {file_path}: {e}")
        return None

def get_all_scores_from_jsonl(file_path: str) -> List:
    """
    Extract all scores from a JSONL file in order, preserving "refused" strings and numeric scores.
    
    Args:
        file_path: Path to the JSONL file
        
    Returns:
        List of scores in order (mix of "refused" strings and numeric values)
    """
    scores = []
    
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            for line in file:
                line = line.strip()
                if line:
                    try:
                        entry = json.loads(line)
                        
                        # Skip metadata entries
                        if 'target_base_url' in entry or 'model_base_url' in entry:
                            continue
                        # Skip goal_achieved entries
                        if 'goal_achieved' in entry:
                            continue
                        
                        # Look for score field
                        if 'score' in entry:
                            score = entry.get('score')
                            # Keep both numeric scores and "refused" strings
                            if isinstance(score, (int, float)) or score == "refused":
                                scores.append(score)
                                
                    except json.JSONDecodeError:
                        continue
                        
    except (FileNotFoundError, Exception) as e:
        print(f"Error reading file {file_path}: {e}")
        
    return scores

In [17]:
def load_batch_data(batch_name: str, tactic: str) -> pd.DataFrame:
    """
    Load data for a specific batch and tactic, filtering for single-turn conversations.
    
    Args:
        batch_name: Name of the batch (e.g., 'batch6A')
        tactic: Jailbreak tactic (e.g., 'direct_request')
        
    Returns:
        DataFrame with file metadata and score sequences
    """
    batch_path = BATCH_PATHS[batch_name]
    data_records = []
    
    # Find all JSONL files
    root_path = Path(batch_path)
    jsonl_files = list(root_path.rglob('*.jsonl'))
    
    print(f"Found {len(jsonl_files)} JSONL files in {batch_path}...")
    
    processed_count = 0
    
    for file_path in jsonl_files:
        metadata = load_jsonl_metadata(str(file_path))
        
        if (metadata and 
            metadata.get('jailbreak_tactic') == tactic and 
            metadata.get('turn_type') == 'single'):
            
            # Get all scores from this file
            scores = get_all_scores_from_jsonl(str(file_path))
            
            if scores:  # Only include files with scores
                record = metadata.copy()
                record['file_path'] = str(file_path)
                record['score_sequence'] = scores
                record['n_attempts'] = len(scores)
                record['batch'] = batch_name
                data_records.append(record)
                processed_count += 1
    
    print(f"Processed {processed_count} files matching criteria ({tactic} + single-turn)")
    
    return pd.DataFrame(data_records)

def extract_first_and_subsequent_responses(df: pd.DataFrame) -> Tuple[List, List]:
    """
    Extract first responses (scratch sampling) and subsequent responses (after refusal) from the data.
    
    Args:
        df: DataFrame with score_sequence column
        
    Returns:
        tuple: (first_responses, subsequent_responses)
               first_responses: list of first response in each conversation
               subsequent_responses: list of all responses after the first in each conversation
    """
    first_responses = []
    subsequent_responses = []
    
    for _, row in df.iterrows():
        scores = row['score_sequence']
        
        if len(scores) > 0:
            # First response (scratch sampling)
            first_responses.append(scores[0])
            
            # Subsequent responses (after refusal) - all responses after the first
            if len(scores) > 1:
                subsequent_responses.extend(scores[1:])
    
    return first_responses, subsequent_responses

def prepare_histogram_data(responses: List) -> Tuple[List[float], int, int]:
    """
    Prepare data for histogram plotting by separating numeric scores and refusals.
    
    Args:
        responses: List of responses (mix of numeric scores and "refused" strings)
        
    Returns:
        tuple: (numeric_scores, refusal_count, total_count)
    """
    numeric_scores = []
    refusal_count = 0
    
    for response in responses:
        if response == "refused":
            refusal_count += 1
        elif isinstance(response, (int, float)):
            numeric_scores.append(response)
    
    total_count = len(responses)
    
    return numeric_scores, refusal_count, total_count

In [18]:
def plot_comparison_histograms(first_numeric, first_refusals, first_total,
                              subsequent_numeric, subsequent_refusals, subsequent_total,
                              title_suffix="", bins=20, figsize=(15, 6)):
    """
    Plot side-by-side histograms comparing first responses vs subsequent responses.
    
    Args:
        first_numeric: Numeric scores from first responses
        first_refusals: Count of refusals in first responses
        first_total: Total first responses
        subsequent_numeric: Numeric scores from subsequent responses
        subsequent_refusals: Count of refusals in subsequent responses
        subsequent_total: Total subsequent responses
        title_suffix: Additional text for plot title
        bins: Number of bins for numeric scores
        figsize: Figure size
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)
    
    # Define bin edges for numeric scores (0 to 1)
    bin_edges = np.linspace(0, 1, bins + 1)
    
    # Plot first responses (scratch sampling)
    if first_numeric:
        counts1, _, _ = ax1.hist(first_numeric, bins=bin_edges, alpha=0.7, color='blue', 
                                edgecolor='black', label=f'Numeric scores (n={len(first_numeric)})')
        max_count1 = max(counts1) if len(counts1) > 0 else 0
    else:
        counts1 = []
        max_count1 = 0
    
    # Add refusal bar at x=1.1 for first responses
    if first_refusals > 0:
        ax1.bar(1.1, first_refusals, width=0.05, alpha=0.7, color='red', 
               edgecolor='black', label=f'Refusals (n={first_refusals})')
        ax1.text(1.1, first_refusals + max(1, max_count1 * 0.05), 'Refused', 
                ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    ax1.set_title(f'First Responses (Scratch Sampling){title_suffix}\nTotal: {first_total}', 
                 fontsize=14, fontweight='bold')
    ax1.set_xlabel('StrongREJECT Score', fontsize=12)
    ax1.set_ylabel('Count', fontsize=12)
    ax1.set_xlim(-0.05, 1.2)
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    
    # Plot subsequent responses (after refusal)
    if subsequent_numeric:
        counts2, _, _ = ax2.hist(subsequent_numeric, bins=bin_edges, alpha=0.7, color='green', 
                                edgecolor='black', label=f'Numeric scores (n={len(subsequent_numeric)})')
        max_count2 = max(counts2) if len(counts2) > 0 else 0
    else:
        counts2 = []
        max_count2 = 0
    
    # Add refusal bar at x=1.1 for subsequent responses
    if subsequent_refusals > 0:
        ax2.bar(1.1, subsequent_refusals, width=0.05, alpha=0.7, color='red', 
               edgecolor='black', label=f'Refusals (n={subsequent_refusals})')
        ax2.text(1.1, subsequent_refusals + max(1, max_count2 * 0.05), 'Refused', 
                ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    ax2.set_title(f'Subsequent Responses (After Refusal){title_suffix}\nTotal: {subsequent_total}', 
                 fontsize=14, fontweight='bold')
    ax2.set_xlabel('StrongREJECT Score', fontsize=12)
    ax2.set_ylabel('Count', fontsize=12)
    ax2.set_xlim(-0.05, 1.2)
    ax2.grid(True, alpha=0.3)
    ax2.legend()
    
    plt.tight_layout()
    plt.show()

def print_summary_statistics(first_numeric, first_refusals, first_total,
                           subsequent_numeric, subsequent_refusals, subsequent_total,
                           batch_name, tactic):
    """
    Print detailed summary statistics for the comparison.
    """
    print(f"\n=== SUMMARY STATISTICS: {batch_name.upper()} - {tactic.upper()} ===")
    
    print("\nFirst Responses (Scratch Sampling):")
    if first_numeric:
        print(f"  Numeric scores - Mean: {np.mean(first_numeric):.3f}, Std: {np.std(first_numeric):.3f}")
        print(f"  Numeric scores - Min: {np.min(first_numeric):.3f}, Max: {np.max(first_numeric):.3f}")
    else:
        print(f"  Numeric scores - No numeric scores found")
    print(f"  Refusal rate: {first_refusals}/{first_total} = {first_refusals/first_total*100:.1f}%")
    
    print("\nSubsequent Responses (After Refusal):")
    if subsequent_numeric:
        print(f"  Numeric scores - Mean: {np.mean(subsequent_numeric):.3f}, Std: {np.std(subsequent_numeric):.3f}")
        print(f"  Numeric scores - Min: {np.min(subsequent_numeric):.3f}, Max: {np.max(subsequent_numeric):.3f}")
    else:
        print(f"  Numeric scores - No numeric scores found")
    print(f"  Refusal rate: {subsequent_refusals}/{subsequent_total} = {subsequent_refusals/subsequent_total*100:.1f}%")
    
    # Key findings
    refusal_rate_diff = (subsequent_refusals/subsequent_total - first_refusals/first_total) * 100
    print(f"\nKey Findings:")
    print(f"  1. Refusal rate difference: {refusal_rate_diff:+.1f} percentage points")
    
    # Statistical comparison
    if first_numeric and subsequent_numeric:
        mean_score_diff = np.mean(subsequent_numeric) - np.mean(first_numeric)
        print(f"  2. Mean numeric score difference: {mean_score_diff:+.3f}")
        
        # Mann-Whitney U test (non-parametric)
        statistic, p_value = stats.mannwhitneyu(first_numeric, subsequent_numeric, alternative='two-sided')
        print(f"  3. Mann-Whitney U test: statistic={statistic:.3f}, p-value={p_value:.6f}")
        
        if p_value < 0.05:
            print(f"  4. Statistical significance: YES (p < 0.05)")
        else:
            print(f"  4. Statistical significance: NO (p >= 0.05)")
    elif not first_numeric and subsequent_numeric:
        print(f"  2. Mean numeric score (subsequent only): {np.mean(subsequent_numeric):.3f}")
        print(f"  3. Cannot compare means - no numeric scores in first responses")
    elif first_numeric and not subsequent_numeric:
        print(f"  2. Mean numeric score (first only): {np.mean(first_numeric):.3f}")
        print(f"  3. Cannot compare means - no numeric scores in subsequent responses")
    else:
        print(f"  2. Cannot compare means - no numeric scores in either group")
    
    # Conclusion
    print(f"\nConclusion for {batch_name} - {tactic}:")
    if abs(refusal_rate_diff) < 5 and (not first_numeric or not subsequent_numeric or abs(mean_score_diff) < 0.05):
        print("  The distributions appear similar, suggesting that sampling from scratch")
        print("  is approximately equivalent to sampling after a refusal.")
    else:
        print("  The distributions show notable differences, suggesting that sampling")
        print("  from scratch may not be equivalent to sampling after a refusal.")

In [19]:
def analyze_batch_tactic_combination(batch_name: str, tactic: str):
    """
    Analyze a specific batch-tactic combination.
    
    Args:
        batch_name: Name of the batch (e.g., 'batch6A')
        tactic: Jailbreak tactic (e.g., 'direct_request')
    """
    print(f"\n" + "="*80)
    print(f"ANALYZING: {batch_name.upper()} - {tactic.upper()}")
    print("="*80)
    
    # Load data
    df = load_batch_data(batch_name, tactic)
    
    if len(df) == 0:
        print(f"No data found for {batch_name} - {tactic}")
        return
    
    print(f"\nLoaded {len(df)} conversations")
    print(f"Test cases: {sorted(df['test_case'].unique())}")
    print(f"Target models: {sorted(df['target_model'].unique())}")
    
    # Show score sequence length distribution
    print(f"\nScore sequence lengths:")
    sequence_lengths = df['n_attempts'].value_counts().sort_index()
    for length, count in sequence_lengths.items():
        print(f"  {length} attempts: {count} conversations")
    
    # Extract first and subsequent responses
    first_responses, subsequent_responses = extract_first_and_subsequent_responses(df)
    
    print(f"\nExtracted responses:")
    print(f"  First responses (scratch sampling): {len(first_responses)} responses")
    print(f"  Subsequent responses (after refusal): {len(subsequent_responses)} responses")
    
    # Prepare data for histograms
    first_numeric, first_refusals, first_total = prepare_histogram_data(first_responses)
    subsequent_numeric, subsequent_refusals, subsequent_total = prepare_histogram_data(subsequent_responses)
    
    print(f"\nBreakdown:")
    print(f"  First responses - Numeric: {len(first_numeric)}, Refusals: {first_refusals}, Total: {first_total}")
    print(f"  Subsequent responses - Numeric: {len(subsequent_numeric)}, Refusals: {subsequent_refusals}, Total: {subsequent_total}")
    
    # Create plots
    title_suffix = f" ({batch_name} - {tactic})"
    plot_comparison_histograms(
        first_numeric, first_refusals, first_total,
        subsequent_numeric, subsequent_refusals, subsequent_total,
        title_suffix=title_suffix,
        bins=BINS
    )
    
    # Print statistics
    print_summary_statistics(
        first_numeric, first_refusals, first_total,
        subsequent_numeric, subsequent_refusals, subsequent_total,
        batch_name, tactic
    )

In [20]:
# COMBINED ANALYSIS - 5th Plot\nprint(\"\\n\" + \"#\"*100)\nprint(\"RUNNING COMBINED ANALYSIS OF ALL 4 CASES\")\nprint(\"#\"*100)\n\n# Collect data from all combinations\nall_first_responses = []\nall_subsequent_responses = []\ncombination_info = []\n\nfor batch_name in ['batch6A', 'batch6B']:\n    for tactic in TACTICS:\n        try:\n            df = load_batch_data(batch_name, tactic)\n            if len(df) == 0:\n                print(f\"No data found for {batch_name} - {tactic}\")\n                continue\n                \n            first_responses, subsequent_responses = extract_first_and_subsequent_responses(df)\n            \n            # Add to combined lists\n            all_first_responses.extend(first_responses)\n            all_subsequent_responses.extend(subsequent_responses)\n            \n            # Track combination info\n            combination_info.append({\n                'batch': batch_name,\n                'tactic': tactic,\n                'conversations': len(df),\n                'first_responses': len(first_responses),\n                'subsequent_responses': len(subsequent_responses)\n            })\n            \n            print(f\"Added {batch_name} - {tactic}: {len(df)} conversations, {len(first_responses)} first, {len(subsequent_responses)} subsequent\")\n            \n        except Exception as e:\n            print(f\"Error loading {batch_name} - {tactic}: {e}\")\n            continue\n\nif all_first_responses:\n    print(f\"\\nCombined totals:\")\n    print(f\"  Total combinations processed: {len(combination_info)}\")\n    print(f\"  Total first responses: {len(all_first_responses)}\")\n    print(f\"  Total subsequent responses: {len(all_subsequent_responses)}\")\n    \n    # Prepare combined data for histograms\n    first_numeric, first_refusals, first_total = prepare_histogram_data(all_first_responses)\n    subsequent_numeric, subsequent_refusals, subsequent_total = prepare_histogram_data(all_subsequent_responses)\n    \n    print(f\"\\nCombined breakdown:\")\n    print(f\"  First responses - Numeric: {len(first_numeric)}, Refusals: {first_refusals}, Total: {first_total}\")\n    print(f\"  Subsequent responses - Numeric: {len(subsequent_numeric)}, Refusals: {subsequent_refusals}, Total: {subsequent_total}\")\n    \n    # Create combined plot (5th plot)\n    title_suffix = \" (Combined All Cases)\"\n    plot_comparison_histograms(\n        first_numeric, first_refusals, first_total,\n        subsequent_numeric, subsequent_refusals, subsequent_total,\n        title_suffix=title_suffix,\n        bins=BINS,\n        figsize=(16, 7)  # Slightly larger for combined plot\n    )\n    \n    # Print combined statistics\n    print_summary_statistics(\n        first_numeric, first_refusals, first_total,\n        subsequent_numeric, subsequent_refusals, subsequent_total,\n        \"Combined\", \"All Cases\"\n    )\n    \n    # Show breakdown by combination\n    print(f\"\\n=== BREAKDOWN BY COMBINATION ===\")\n    combination_df = pd.DataFrame(combination_info)\n    print(combination_df.to_string(index=False))\n    \n    # Overall conclusion\n    print(\"\\n\" + \"=\"*80)\n    print(\"OVERALL CONCLUSION\")\n    print(\"=\"*80)\n    \n    first_refusal_rate = first_refusals / first_total * 100\n    subsequent_refusal_rate = subsequent_refusals / subsequent_total * 100\n    refusal_rate_diff = subsequent_refusal_rate - first_refusal_rate\n    \n    print(f\"\\nAcross all combinations ({len(combination_info)} total):\")\n    print(f\"  Total conversations analyzed: {sum(info['conversations'] for info in combination_info)}\")\n    print(f\"  Total first responses: {first_total}\")\n    print(f\"  Total subsequent responses: {subsequent_total}\")\n    print(f\"  First refusal rate: {first_refusal_rate:.1f}%\")\n    print(f\"  Subsequent refusal rate: {subsequent_refusal_rate:.1f}%\")\n    print(f\"  Difference: {refusal_rate_diff:+.1f} percentage points\")\n    \n    if first_numeric and subsequent_numeric:\n        first_mean = np.mean(first_numeric)\n        subsequent_mean = np.mean(subsequent_numeric)\n        mean_diff = subsequent_mean - first_mean\n        print(f\"  First numeric mean: {first_mean:.3f}\")\n        print(f\"  Subsequent numeric mean: {subsequent_mean:.3f}\")\n        print(f\"  Mean difference: {mean_diff:+.3f}\")\n        \n        # Overall statistical test\n        _, p_value = stats.mannwhitneyu(\n            first_numeric, subsequent_numeric, alternative='two-sided'\n        )\n        print(f\"  Combined p-value: {p_value:.6f}\")\n        \n        if p_value < 0.05:\n            print(f\"\\n🔍 RESULT: STATISTICALLY SIGNIFICANT difference found (p < 0.05)\")\n            print(f\"   Sampling from scratch is NOT equivalent to sampling after refusal.\")\n        else:\n            print(f\"\\n✅ RESULT: NO statistically significant difference found (p >= 0.05)\")\n            print(f\"   Sampling from scratch appears equivalent to sampling after refusal.\")\n    \n    print(f\"\\nThe combined analysis provides the most robust comparison by pooling\")\n    print(f\"data from all experimental conditions (tactics and batches) for maximum statistical power.\")\n    \nelse:\n    print(\"No data was successfully processed for combined analysis.\")"

In [21]:
# Comparative summary across all combinations\nprint(\"\\n\" + \"=\"*100)\nprint(\"COMPARATIVE SUMMARY ACROSS ALL COMBINATIONS\")\nprint(\"=\"*100)\n\nsummary_results = []\n\nfor batch_name in ['batch6A', 'batch6B']:\n    for tactic in TACTICS:\n        try:\n            df = load_batch_data(batch_name, tactic)\n            if len(df) == 0:\n                continue\n                \n            first_responses, subsequent_responses = extract_first_and_subsequent_responses(df)\n            first_numeric, first_refusals, first_total = prepare_histogram_data(first_responses)\n            subsequent_numeric, subsequent_refusals, subsequent_total = prepare_histogram_data(subsequent_responses)\n            \n            # Calculate key metrics\n            first_refusal_rate = first_refusals / first_total * 100 if first_total > 0 else 0\n            subsequent_refusal_rate = subsequent_refusals / subsequent_total * 100 if subsequent_total > 0 else 0\n            refusal_rate_diff = subsequent_refusal_rate - first_refusal_rate\n            \n            first_mean = np.mean(first_numeric) if first_numeric else None\n            subsequent_mean = np.mean(subsequent_numeric) if subsequent_numeric else None\n            mean_diff = subsequent_mean - first_mean if (first_mean is not None and subsequent_mean is not None) else None\n            \n            # Statistical test\n            p_value = None\n            if first_numeric and subsequent_numeric:\n                _, p_value = stats.mannwhitneyu(first_numeric, subsequent_numeric, alternative='two-sided')\n            \n            summary_results.append({\n                'batch': batch_name,\n                'tactic': tactic,\n                'n_conversations': len(df),\n                'first_total': first_total,\n                'subsequent_total': subsequent_total,\n                'first_refusal_rate': first_refusal_rate,\n                'subsequent_refusal_rate': subsequent_refusal_rate,\n                'refusal_rate_diff': refusal_rate_diff,\n                'first_mean': first_mean,\n                'subsequent_mean': subsequent_mean,\n                'mean_diff': mean_diff,\n                'p_value': p_value\n            })\n            \n        except Exception as e:\n            print(f\"Error processing {batch_name} - {tactic}: {e}\")\n            continue\n\n# Create summary DataFrame and display\nif summary_results:\n    summary_df = pd.DataFrame(summary_results)\n    \n    print(\"\\nSummary Table:\")\n    pd.set_option('display.max_columns', None)\n    pd.set_option('display.width', None)\n    print(summary_df.round(3).to_string(index=False))\n    \n    print(\"\\nKey Insights:\")\n    print(f\"1. Average refusal rate difference across all combinations: {summary_df['refusal_rate_diff'].mean():.1f} percentage points\")\n    \n    significant_combinations = summary_df[summary_df['p_value'] < 0.05] if 'p_value' in summary_df.columns else pd.DataFrame()\n    if len(significant_combinations) > 0:\n        print(f\"2. Combinations with statistically significant differences: {len(significant_combinations)}/{len(summary_df)}\")\n        for _, row in significant_combinations.iterrows():\n            print(f\"   - {row['batch']} {row['tactic']}: p = {row['p_value']:.6f}\")\n    else:\n        print(f\"2. No combinations showed statistically significant differences in numeric scores\")\n    \n    print(f\"3. Range of refusal rate differences: {summary_df['refusal_rate_diff'].min():.1f} to {summary_df['refusal_rate_diff'].max():.1f} percentage points\")\nelse:\n    print(\"No data processed for summary.\")"